# Equity Duration × ECB Monetary Policy Shocks

This notebook implements an event-study design to quantify whether **equity duration** amplifies firm-level stock price reactions to **ECB monetary policy shocks**.

## Overview and Research Question

This notebook tests whether **high-duration equities react more strongly** to ECB monetary policy surprises than **low-duration equities**.  
I implement both (i) **continuous interaction regressions** and (ii) **portfolio-split regressions** (High vs Low duration).

Throughout the notebook I compare two equity-duration measures:

- **`Duration_CAPM`**: firm-specific discounting using a CAPM-implied cost of equity.
- **`Duration_r125`**: discounting all firms at a constant 12.5% rate (pure cash-flow timing).

The empirical focus is on ECB meeting days and two orthogonal shocks:

- **`ShockMP`**: “pure” monetary policy shock (discount-rate channel).
- **`ShockInfo`**: central bank information shock (cash-flow/news channel).

## Empirical idea (event-study with cross-sectional heterogeneity)

- ECB announcements define a set of **event dates**.
- On each event date, I compute **firm-level abnormal returns** (AR).
- I regress AR on ECB shocks and their **interaction with equity duration**.
- Standard errors are **clustered by event date**, which is the appropriate level given common shocks at each ECB meeting.

The main coefficient of interest is the interaction term:

$$
AR_{i,t} = \beta_0 + \beta_1 Shock_t + \beta_2 D_i + \beta_3 (Shock_t \times D_i) + \Gamma'X_i + \varepsilon_{i,t}
$$

where \(D_i\) is (standardized) equity duration and \(X_i\) are firm controls (e.g., size and beta).


In [37]:
# Core
import numpy as np
import pandas as pd
from pathlib import Path
# Stats / regressions
import statsmodels.formula.api as smf
# Plots
import matplotlib.pyplot as plt

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

DUR_FILE = TABLE_DIR / "final_results_duration.xlsx"
SHOCK_FILE = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## 2. Load firm-level equity duration and characteristics

I load the firm-level duration measures and keep **Duration (CAPM r)** as the baseline equity-duration proxy.


In [38]:
# Load durations (time-varying panel from EQDuration)

df_dur_panel = load_parquet("durations_panel")  # columns: RIC | YEAR | Duration_* | D_emp_2yOIS

# Basic cleaning
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# Choose baseline duration measure for the main analysis (now from panel)
DUR_COL = "Duration_CAPM"          # or: "Duration_r125", "Duration_undiscounted"
df_dur_panel[DUR_COL] = pd.to_numeric(df_dur_panel[DUR_COL], errors="coerce")

print("Durations panel shape:", df_dur_panel.shape)
display(df_dur_panel[["RIC", "YEAR", DUR_COL]].head())
print(df_dur_panel[DUR_COL].describe())

Durations panel shape: (18009, 6)


,RIC,YEAR,Duration_CAPM
0,1COVG.DE^L25,1999,NaN
1,1COVG.DE^L25,2000,NaN
2,1COVG.DE^L25,2001,NaN
3,1COVG.DE^L25,2002,NaN
4,1COVG.DE^L25,2003,NaN


count     9802.000000
mean         7.068650
std        175.703944
min     -16980.793418
25%          6.853186
50%         11.932390
75%         15.727115
max       1038.038250
Name: Duration_CAPM, dtype: float64


## 3. Load ECB shock series


### Economic Meaning and Sign Conventions

I use the Jarociński–Karadi (2020) decomposition of high-frequency ECB surprises into two orthogonal components:

- **`ShockMP` (Monetary Policy Shock):** the “pure” policy component.  
  A **positive** value corresponds to an unexpected **tightening** (higher-than-expected short rates).  
  **Prediction:** on average, equity prices fall → abnormal returns should be negative.

- **`ShockInfo` (Information Shock):** captures the information revealed by the central bank about the outlook.  
  A **positive** value is interpreted as **good news** about fundamentals (often associated with rising yields and rising equities).  
  **Prediction:** equity prices may rise, depending on the balance of improved cash-flow expectations vs discount-rate effects.

I run both a one-shock (MP only) specification and a two-shock specification to separately identify discount-rate vs information channels.

In [39]:
# --- 3. Load ECB monetary policy shocks (Jarocinski & Karadi) ---

df_shk = pd.read_csv(SHOCK_FILE)

# ------------------------------------------------------------
# 1. Parse dates
# ------------------------------------------------------------
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")

assert df_shk["date"].notna().all(), "Some ECB shock dates could not be parsed."

# ------------------------------------------------------------
# 2. Select shock decomposition
# ------------------------------------------------------------
# Options:
#   - "pm"     : Poor Man's sign restrictions (mutually exclusive shocks)
#   - "median" : Median rotation (both shocks can occur jointly)

SHOCK_VERSION = "pm"    # <-- central switch for the paper

if SHOCK_VERSION == "pm":
    shock_map = {
        "MP_pm": "ShockMP",
        "CBI_pm": "ShockInfo"
    }
elif SHOCK_VERSION == "median":
    shock_map = {
        "MP_median": "ShockMP",
        "CBI_median": "ShockInfo"
    }
else:
    raise ValueError("SHOCK_VERSION must be 'pm' or 'median'.")

missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing required shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)

# ------------------------------------------------------------
# 3. Optional: keep policy factor for diagnostics
# ------------------------------------------------------------
if "pc1" in df_shk.columns:
    df_shk["PolicyFactor"] = pd.to_numeric(df_shk["pc1"], errors="coerce")

# ------------------------------------------------------------
# 4. Keep one observation per announcement date
# ------------------------------------------------------------
df_shk = (
    df_shk
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Numeric coercion
# ------------------------------------------------------------
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

# ------------------------------------------------------------
# 6. Consistency check (theoretical identity)
# ------------------------------------------------------------
if "PolicyFactor" in df_shk.columns:
    check = df_shk["ShockMP"] + df_shk["ShockInfo"] - df_shk["PolicyFactor"]
    print("Max |MP + Info − pc1|:", check.abs().max())

# ------------------------------------------------------------
# 7. Final sanity output
# ------------------------------------------------------------
print("Shock version:", SHOCK_VERSION)
print("Shocks shape:", df_shk.shape)

display(df_shk[["date", "ShockMP", "ShockInfo"]].head())
display(df_shk[["ShockMP", "ShockInfo"]].describe())

Max |MP + Info − pc1|: 0.0
Shock version: pm
Shocks shape: (312, 8)


,date,ShockMP,ShockInfo
0,1999-01-07,-0.000000,-0.037546
1,1999-01-21,0.003581,0.000000
2,1999-02-18,-0.000000,-0.000000
3,1999-03-04,-0.001926,-0.000000
4,1999-03-18,-0.000758,-0.000000


,ShockMP,ShockInfo
count,312.000000,312.000000
mean,0.002615,0.000574
std,0.034716,0.026557
min,-0.233329,-0.163802
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.002158,0.000000
max,0.203563,0.200709


## 4. Event Panel Shocks x Duratons

### Load daily returns and construct abnormal returns (AR)

My baseline dependent variable is the **abnormal return** on each firm-event date.  
(Depending on the earlier construction in the notebook, this is either a market-adjusted return or a model-based abnormal return.)

To avoid relying on external index data (e.g., STOXX), I compute the daily market return directly from my sample:

$$
mkt\_ret_t = \frac{1}{N_t}\sum_i r_{i,t}
$$

and define abnormal returns as:

$$
AR_{i,t} = r_{i,t} - mkt\_ret_t
$$

This is a standard equal-weighted benchmark in event-study applications.

### Build the event-day panel (firm × ECB event date)

I keep only returns observed on ECB event dates and merge:
- shocks by `date`
- duration and firm controls by `RIC`

This creates the core event-study panel used in the regressions.

### Event Windows
- **Day 0:** the ECB announcement day (baseline window).
- **[0,+1] window:** a robustness check that sums Day 0 and Day +1 abnormal returns to capture delayed price adjustment.

Interpretation: the shock occurs on Day 0, but markets may incorporate information with a short delay, so [0,+1] provides a more conservative reaction measure.

### Standardize equity duration

I standardize duration to mean 0 and standard deviation 1:

$$
D^{std}_i = \frac{D_i - \bar{D}}{sd(D)}
$$

This makes interaction coefficients easy to interpret as the effect per **one standard deviation** increase in duration.



In [40]:
# ============================================================
# Build firm × event-date panel and merge ECB shocks + durations
# + standardize each duration (new *_std columns)
# ============================================================

import pandas as pd

# --- 1) Event dates from shocks ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

event_dates = df_shk["date"].dropna().sort_values().unique()
print("Number of event dates:", len(event_dates))

# --- 2) Detect date column in df_ret ---
if "date" in df_ret.columns:
    DATE_COL = "date"
else:
    candidates = [c for c in df_ret.columns if c.lower() in ["day", "tradedate", "event_date", "datadate", "datetime"]]
    if len(candidates) == 0:
        raise ValueError(
            "Could not find a date column in df_ret. "
            "Rename your date column to 'date' or set DATE_COL manually."
        )
    DATE_COL = candidates[0]

print("Using df_ret date column:", DATE_COL)

# Parse dates in df_ret
df_ret[DATE_COL] = pd.to_datetime(df_ret[DATE_COL], errors="coerce")
assert df_ret[DATE_COL].notna().any(), "All df_ret dates are NaT after parsing."

# --- 3) Build event panel ---
df_evt = df_ret[df_ret[DATE_COL].isin(event_dates)].copy()
df_evt = df_evt.rename(columns={DATE_COL: "date"})
df_evt["RIC"] = df_evt["RIC"].astype(str).str.strip()

print("df_evt shape (firm × event dates):", df_evt.shape)
display(df_evt.head())

# --- 4) Merge shocks ---
df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)

miss_shk = df_evt[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 5) Construct predetermined YEAR for duration merge (use year-end t-1) ---
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# --- 6) Merge durations ---
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# NOTE: You changed this to startswith("D"). We'll keep that but ensure YEAR/RIC not accidentally included.
duration_cols = [c for c in df_dur_panel.columns if c.startswith("D") and c not in ["DATE", "DAY"]]

if len(duration_cols) == 0:
    raise ValueError("No duration columns found in df_dur_panel (expected columns starting with 'D' or 'Duration_').")

# Ensure numeric
for c in duration_cols:
    df_dur_panel[c] = pd.to_numeric(df_dur_panel[c], errors="coerce")

df_evt = df_evt.merge(
    df_dur_panel[["RIC", "YEAR"] + duration_cols],
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

print("df_evt shape after merging shocks + durations:", df_evt.shape)

# --- 6b) Standardize each duration into a separate *_std column (z-score) ---
# Standardization is done on the merged sample (df_evt), ignoring NaNs.
# If you prefer standardization across the full durations panel instead, tell me and I’ll adjust.

std_cols = []
for c in duration_cols:
    mu = df_evt[c].mean(skipna=True)
    sd = df_evt[c].std(skipna=True, ddof=0)  # ddof=0 -> population std; use ddof=1 if you prefer sample std
    new_c = f"{c}_std"
    std_cols.append(new_c)

    if pd.isna(sd) or sd == 0:
        df_evt[new_c] = pd.NA
    else:
        df_evt[new_c] = (df_evt[c] - mu) / sd

print("Standardized duration columns added:", std_cols)

# --- 7) Diagnostics ---
print("\nMissing share by duration column:")
display(df_evt[duration_cols].isna().mean().sort_values())

print("\nMissing share by standardized duration column:")
display(df_evt[std_cols].isna().mean().sort_values())

print("\nPreview merged columns:")
display(df_evt[["RIC", "date", "YEAR", "ShockMP", "ShockInfo"] + duration_cols + std_cols].head(10))

Number of event dates: 312
Using df_ret date column: date
df_evt shape (firm × event dates): (136842, 5)


,date,RIC,ret,mkt_ret,AR
11,2015-10-22,1COVG.DE^L25,3.585657,1.34795,2.237708
41,2015-12-03,1COVG.DE^L25,-2.86533,-2.109469,-0.755861
72,2016-01-21,1COVG.DE^L25,3.97295,1.814211,2.158739
107,2016-03-10,1COVG.DE^L25,-0.812586,-0.921107,0.10852
135,2016-04-21,1COVG.DE^L25,1.04227,-0.120709,1.162978


Share of rows with missing shocks: 0.0
df_evt shape after merging shocks + durations: (136842, 12)
Standardized duration columns added: ['Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std', 'D_emp_2yOIS_std']

Missing share by duration column:


D_emp_2yOIS              0.115674
Duration_r125            0.203286
Duration_undiscounted    0.204236
Duration_CAPM            0.289158
dtype: float64


Missing share by standardized duration column:


D_emp_2yOIS_std              0.115674
Duration_r125_std            0.203286
Duration_undiscounted_std    0.204236
Duration_CAPM_std            0.289158
dtype: float64


Preview merged columns:


,RIC,date,YEAR,ShockMP,ShockInfo,Duration_undiscounted,Duration_r125,Duration_CAPM,D_emp_2yOIS,Duration_undiscounted_std,Duration_r125_std,Duration_CAPM_std,D_emp_2yOIS_std
0,1COVG.DE^L25,2015-10-22,2014,-0.020880,-0.000000,NaN,NaN,NaN,-1.906057,NaN,NaN,NaN,0.668455
1,1COVG.DE^L25,2015-12-03,2014,0.097447,0.000000,NaN,NaN,NaN,-1.906057,NaN,NaN,NaN,0.668455
2,1COVG.DE^L25,2016-01-21,2015,-0.021455,-0.000000,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
3,1COVG.DE^L25,2016-03-10,2015,0.000000,0.049913,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
4,1COVG.DE^L25,2016-04-21,2015,0.001604,0.000000,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
5,1COVG.DE^L25,2016-06-02,2015,-0.000000,-0.000530,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
6,1COVG.DE^L25,2016-07-21,2015,0.000000,0.025576,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
7,1COVG.DE^L25,2016-09-08,2015,0.013409,0.000000,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
8,1COVG.DE^L25,2016-10-20,2015,-0.000548,-0.000000,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455
9,1COVG.DE^L25,2016-12-08,2015,0.000000,0.001250,2.549836,6.988085,NaN,-1.906057,-0.098777,0.005419,NaN,0.668455


## Why I Use Predetermined Duration (YEAR − 1)

To avoid **look-ahead bias**, I merge **firm-year duration measures from the previous year** to each ECB event.  
Concretely, for an ECB announcement in calendar year *t*, I use the equity duration measured at **year-end (12/31) of year *t−1***.

This ensures that duration is **known before the event** and cannot be mechanically affected by event-day returns.

**Implementation:**  
`YEAR = year(date) − 1` and merge on (`RIC`, `YEAR`).

## 7. Baseline regression: monetary policy shock × duration

### Core Hypotheses and Expected Signs

#### Continuous interaction regressions
I estimate regressions of the form:

$$
AR_{i,t} = \alpha + \beta_1 ShockMP_t + \beta_2 D_{i,t-1} + \beta_3 (ShockMP_t \times D_{i,t-1}) + \varepsilon_{i,t}
$$

and in the two-shock case additionally include `ShockInfo` and `ShockInfo × Duration`.

**Expected signs (baseline intuition):**
- **MP shocks:** if `ShockMP > 0` is a tightening, then higher-duration firms should be more sensitive to discount-rate increases  
  → **expect \(\beta_3 < 0\)** (more negative AR for higher duration).
- **Info shocks:** ambiguous ex ante. If `ShockInfo > 0` reflects positive fundamental news, long-duration (growth) firms may react more  
  → the sign of `ShockInfo × Duration` is an empirical question.

#### Portfolio-split regressions (non-linearity)
Because duration effects may be **non-linear**, I also estimate a High vs Low split:

- Sort firms into terciles by duration (within each year, using predetermined duration).
- Keep only **Low** and **High** terciles.
- Define `HighDur = 1` for High tercile, `0` for Low tercile.

Regression:

$$
AR_{i,t} = \alpha + \gamma_1 Shock_t + \gamma_2 (Shock_t \times HighDur_i) + \varepsilon_{i,t}
$$

Here, `Shock × HighDur` directly measures how differently high-duration firms respond compared to low-duration firms.

#### Comparing Duration_CAPM vs Duration_r125

I run the same specifications using both duration measures:

- **`Duration_CAPM`** incorporates firm-specific discount rates (risk premia), so high-beta firms may mechanically have lower duration.
- **`Duration_r125`** uses a common discount rate, capturing mostly cash-flow timing.

Comparing results across the two measures helps distinguish whether heterogeneity is driven mainly by:
1) **cash-flow timing** (r125), or  
2) **risk-adjusted discounting** (CAPM).

In the results, I therefore report side-by-side estimates for **CAPM** and **r125** durations.

#### Inference

All regressions use **event-date clustered standard errors**, because residuals are correlated across firms on the same ECB announcement day (common shock).

This is the appropriate inference design for firm×event panels with a single shock per event date.

#### Notes on Interpretation

The key objects are:
- `ShockMP × Duration` (or `ShockMP × HighDur`): discount-rate sensitivity heterogeneity
- `ShockInfo × Duration` (or `ShockInfo × HighDur`): information / cash-flow news heterogeneity

Even if average effects are weak, significant differences between High and Low duration groups indicate heterogeneous monetary policy transmission across equities.

In [41]:
# ============================================================
# 7. Baseline regression (updated for time-varying durations)
# ============================================================

import statsmodels.formula.api as smf

# --- 0) Choose baseline standardized duration ---
# Pick ONE of the standardized duration columns created earlier
BASELINE_DUR_STD = "Duration_CAPM_std"   # e.g. "Duration_r125_std", "Duration_undiscounted_std"

assert BASELINE_DUR_STD in df_evt.columns, f"{BASELINE_DUR_STD} not found in df_evt."

# --- 1) Build regression sample ---
# IMPORTANT: drop missing values explicitly BEFORE clustering
reg_cols = ["AR", "ShockMP", BASELINE_DUR_STD, "date"]

if "ln_mktcap" in df_evt.columns:
    reg_cols.append("ln_mktcap")

# handle space in column name safely via Q()
HAS_BETA = 'Beta (5Y)' in df_evt.columns
if HAS_BETA:
    reg_cols.append("Beta (5Y)")

df_reg = df_evt[reg_cols].copy().dropna()

print("Regression sample:", df_reg.shape)
print("Unique event dates:", df_reg["date"].nunique())

# --- 2) Regression formula ---
formula = f'AR ~ ShockMP * {BASELINE_DUR_STD}'

if "ln_mktcap" in df_reg.columns:
    formula += " + ln_mktcap"

if HAS_BETA:
    formula += ' + Q("Beta (5Y)")'

print("Formula:", formula)

# --- 3) Estimate with event-date clustered SEs ---
m1 = smf.ols(formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["date"]}
)

print(m1.summary())

Regression sample: (97273, 4)
Unique event dates: 265
Formula: AR ~ ShockMP * Duration_CAPM_std
                            OLS Regression Results                            
Dep. Variable:                     AR   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                    0.8150
Date:                Wed, 28 Jan 2026   Prob (F-statistic):              0.487
Time:                        13:33:39   Log-Likelihood:            -2.2661e+05
No. Observations:               97273   AIC:                         4.532e+05
Df Residuals:                   97269   BIC:                         4.533e+05
Df Model:                           3                                         
Covariance Type:              cluster                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
--------------------

## 8. Two-shock regression: policy shock vs. information shock

To separate the rate component from information effects, I estimate:

$$
AR_{i,t} = \beta_0 + \beta_1 ShockMP_t + \beta_2 ShockInfo_t + 
\beta_3 (ShockMP_t \times D_i^{std}) + \beta_4 (ShockInfo_t \times D_i^{std})
+ \Gamma'X_i + \varepsilon_{i,t}
$$

The key coefficients are the interaction terms.


In [42]:
# ============================================================
# 8. Two-shock regression (MP + Information shocks)
# ============================================================

import statsmodels.formula.api as smf

# --- 0) Choose baseline standardized duration ---
BASELINE_DUR_STD = "Duration_CAPM_std"   # switch if needed

assert BASELINE_DUR_STD in df_evt.columns, f"{BASELINE_DUR_STD} not found in df_evt."

# --- 1) Build regression sample ---
reg_cols2 = ["AR", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date"]

if "ln_mktcap" in df_evt.columns:
    reg_cols2.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt.columns
if HAS_BETA:
    reg_cols2.append("Beta (5Y)")

df_reg2 = df_evt[reg_cols2].copy().dropna()

print("Regression sample (two-shock):", df_reg2.shape)
print("Unique event dates:", df_reg2["date"].nunique())

# --- 2) Regression formula ---
formula2 = (
    f'AR ~ ShockMP * {BASELINE_DUR_STD} '
    f'+ ShockInfo * {BASELINE_DUR_STD}'
)

if "ln_mktcap" in df_reg2.columns:
    formula2 += " + ln_mktcap"

if HAS_BETA:
    formula2 += ' + Q("Beta (5Y)")'

print("Formula:", formula2)

# --- 3) Estimate with event-date clustered SEs ---
m2 = smf.ols(formula2, data=df_reg2).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg2["date"]}
)

print(m2.summary())

Regression sample (two-shock): (97273, 5)
Unique event dates: 265
Formula: AR ~ ShockMP * Duration_CAPM_std + ShockInfo * Duration_CAPM_std
                            OLS Regression Results                            
Dep. Variable:                     AR   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     1.112
Date:                Wed, 28 Jan 2026   Prob (F-statistic):              0.354
Time:                        13:33:39   Log-Likelihood:            -2.2543e+05
No. Observations:               97273   AIC:                         4.509e+05
Df Residuals:                   97267   BIC:                         4.509e+05
Df Model:                           5                                         
Covariance Type:              cluster                                         
                                  coef    std err          z      P>|z

## 9. Robustness: [0, +1] event window (two-day cumulative AR)

As a robustness check, I compute the two-day cumulative abnormal return for each firm around the ECB announcement:

$$
AR^{0,1}_{i,t} = AR_{i,t} + AR_{i,t+1}
$$

**Implementation detail:** I use the next *trading day* per firm by shifting within each firm (`groupby("RIC")`).
Missing leads (e.g., end-of-sample or delistings) naturally reduce the sample; I drop NAs explicitly before estimation to avoid clustering errors.


In [43]:
# ============================================================
# 9. [0,+1] abnormal return window (FIXED: no 1:1 merge needed)
# - builds AR_01
# - merges shocks
# - merges time-varying durations (RIC x YEAR)
# - standardizes durations within df_evt_01 -> new *_std columns
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --- 0) Build AR[0,+1] at firm-day level ---
df_ret_sorted = df_ret.sort_values(["RIC", "date"]).copy()
df_ret_sorted["date"] = pd.to_datetime(df_ret_sorted["date"], errors="coerce")

df_ret_sorted["AR0"] = df_ret_sorted["AR"]
df_ret_sorted["AR1"] = df_ret_sorted.groupby("RIC")["AR0"].shift(-1)
df_ret_sorted["AR_01"] = df_ret_sorted["AR0"] + df_ret_sorted["AR1"]

# --- 1) Keep only ECB event dates (announcement day = day 0) ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
event_dates = df_shk["date"].dropna().unique()

df_evt_01 = df_ret_sorted[df_ret_sorted["date"].isin(event_dates)].copy()
df_evt_01["RIC"] = df_evt_01["RIC"].astype(str).str.strip()

print("Event-panel [0,+1] shape (before merges):", df_evt_01.shape)

# --- 2) Merge shocks (m:1) ---
df_evt_01 = df_evt_01.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)

miss_shk = df_evt_01[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 3) Construct predetermined YEAR and merge durations (m:1) ---
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# durations: prefer Duration_*; fallback to columns starting with "D" if needed
duration_cols = [c for c in df_dur_panel.columns if c.startswith("Duration_")]
if len(duration_cols) == 0:
    duration_cols = [c for c in df_dur_panel.columns if c.startswith("D") and c not in ["DATE", "DAY"]]

if len(duration_cols) == 0:
    raise ValueError("No duration columns found in df_dur_panel (expected 'Duration_*' or 'D*').")

for c in duration_cols:
    df_dur_panel[c] = pd.to_numeric(df_dur_panel[c], errors="coerce")

df_evt_01 = df_evt_01.merge(
    df_dur_panel[["RIC", "YEAR"] + duration_cols],
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

# --- 4) Controls ---
if "MarketCap" in df_evt_01.columns:
    df_evt_01["ln_mktcap"] = np.log(df_evt_01["MarketCap"])

# --- 5) Standardize durations IN THIS PANEL (new *_std columns) ---
std_cols = []
for c in duration_cols:
    mu = df_evt_01[c].mean(skipna=True)
    sd = df_evt_01[c].std(skipna=True, ddof=0)  # ddof=0; use ddof=1 if you prefer
    new_c = f"{c}_std"
    std_cols.append(new_c)

    if pd.isna(sd) or sd == 0:
        df_evt_01[new_c] = pd.NA
    else:
        df_evt_01[new_c] = (df_evt_01[c] - mu) / sd

print("Standardized duration columns added:", std_cols)

# --- 6) Regression-ready sample ---
BASELINE_DUR_STD = "Duration_CAPM_std"  # switch if needed
if BASELINE_DUR_STD not in df_evt_01.columns:
    raise ValueError(f"{BASELINE_DUR_STD} not found. Available std cols: {std_cols[:10]} ...")

reg_cols_01 = ["AR_01", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date"]

if "ln_mktcap" in df_evt_01.columns:
    reg_cols_01.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt_01.columns
if HAS_BETA:
    reg_cols_01.append("Beta (5Y)")

df_reg_01 = df_evt_01[reg_cols_01].copy().dropna()

print("Regression sample [0,+1]:", df_reg_01.shape)
print("Unique event dates:", df_reg_01["date"].nunique())

# --- 7) Two-shock regression on AR[0,+1] ---
formula_01 = (
    f'AR_01 ~ ShockMP * {BASELINE_DUR_STD} '
    f'+ ShockInfo * {BASELINE_DUR_STD}'
)

if "ln_mktcap" in df_reg_01.columns:
    formula_01 += " + ln_mktcap"

if HAS_BETA:
    formula_01 += ' + Q("Beta (5Y)")'

print("Formula:", formula_01)

m2_01 = smf.ols(formula_01, data=df_reg_01).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_01["date"]}
)

print(m2_01.summary())

Event-panel [0,+1] shape (before merges): (136842, 8)
Share of rows with missing shocks: 0.0
Standardized duration columns added: ['Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std']
Regression sample [0,+1]: (97267, 5)
Unique event dates: 265
Formula: AR_01 ~ ShockMP * Duration_CAPM_std + ShockInfo * Duration_CAPM_std
                            OLS Regression Results                            
Dep. Variable:                  AR_01   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                    0.9523
Date:                Wed, 28 Jan 2026   Prob (F-statistic):              0.448
Time:                        13:33:39   Log-Likelihood:            -2.5273e+05
No. Observations:               97267   AIC:                         5.055e+05
Df Residuals:                   97261   BIC:                         5.055e+05
Df Model:                   

In [44]:
# ============================================================
# Portfolio split (robust): year-by-year qcut with safeguards
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

BASE_DUR = "Duration_CAPM"   # or another raw duration column
assert BASE_DUR in df_evt.columns, f"{BASE_DUR} not found in df_evt."

def safe_qcut_terciles(x: pd.Series):
    """
    Return tercile labels for a Series x (same index).
    If not enough valid/unique values, return NA for the whole group.
    """
    x_valid = x.dropna()

    # Need at least 3 obs and at least 3 distinct values to form terciles reliably
    if (len(x_valid) < 20) or (x_valid.nunique() < 3):
        return pd.Series(pd.NA, index=x.index, dtype="object")

    try:
        # duplicates='drop' avoids error when some quantile edges coincide
        labels = pd.qcut(x_valid, q=3, labels=["Low", "Mid", "High"], duplicates="drop")
        out = pd.Series(pd.NA, index=x.index, dtype="object")
        out.loc[x_valid.index] = labels.astype("object")
        return out
    except Exception:
        return pd.Series(pd.NA, index=x.index, dtype="object")

# 1) Build duration groups year-by-year (predetermined)
df_evt["Dur_grp"] = df_evt.groupby("YEAR")[BASE_DUR].transform(safe_qcut_terciles)

# Diagnostics: how many got assigned?
print("Dur_grp value counts (including NA):")
display(df_evt["Dur_grp"].value_counts(dropna=False))

# Optional: show years with missing assignment
yrs_bad = (
    df_evt.groupby("YEAR")["Dur_grp"]
    .apply(lambda s: s.notna().mean())
    .sort_values()
)
print("Share assigned per YEAR (lowest first):")
display(yrs_bad.head(10))

# 2) Keep only Low vs High (strongest contrast)
df_evt_hl = df_evt[df_evt["Dur_grp"].isin(["Low", "High"])].copy()
df_evt_hl["HighDur"] = (df_evt_hl["Dur_grp"] == "High").astype(int)

print("HL sample shape:", df_evt_hl.shape)
print("HL groups:")
display(df_evt_hl["Dur_grp"].value_counts())

# 3) Portfolio-split regression (Two-shock)
reg_cols = ["AR", "ShockMP", "ShockInfo", "HighDur", "date"]
if "ln_mktcap" in df_evt_hl.columns:
    reg_cols.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt_hl.columns
if HAS_BETA:
    reg_cols.append("Beta (5Y)")

df_reg_ps = df_evt_hl[reg_cols].dropna()

print("Regression sample (portfolio split):", df_reg_ps.shape)
print("Unique event dates:", df_reg_ps["date"].nunique())

formula = "AR ~ ShockMP + ShockInfo + ShockMP:HighDur + ShockInfo:HighDur"
if "ln_mktcap" in df_reg_ps.columns:
    formula += " + ln_mktcap"
if HAS_BETA:
    formula += ' + Q("Beta (5Y)")'

print("Formula:", formula)

m_ps = smf.ols(formula, data=df_reg_ps).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_ps["date"]}
)

print(m_ps.summary())

Dur_grp value counts (including NA):


Dur_grp
<NA>    39569
Low     32546
Mid     32400
High    32327
Name: count, dtype: int64

Share assigned per YEAR (lowest first):


YEAR
1998    0.000000
1999    0.000000
2000    0.679845
2005    0.754187
2006    0.761500
2004    0.771549
2001    0.772084
2002    0.788619
2007    0.803030
2003    0.807692
Name: Dur_grp, dtype: float64

HL sample shape: (64873, 18)
HL groups:


Dur_grp
Low     32546
High    32327
Name: count, dtype: int64

Regression sample (portfolio split): (64873, 5)
Unique event dates: 265
Formula: AR ~ ShockMP + ShockInfo + ShockMP:HighDur + ShockInfo:HighDur
                            OLS Regression Results                            
Dep. Variable:                     AR   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     2.344
Date:                Wed, 28 Jan 2026   Prob (F-statistic):             0.0552
Time:                        13:33:40   Log-Likelihood:            -1.5102e+05
No. Observations:               64873   AIC:                         3.020e+05
Df Residuals:                   64868   BIC:                         3.021e+05
Df Model:                           4                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|     